In [ ]:
# Retail Sales & Order Operations Analytics

## Data Cleaning and Preparation

This notebook prepares the Olist e-commerce datasets for analysis
in MySQL and Power BI.

The cleaning process includes:

- Loading relational CSV files
- Inspecting rows, columns, missing values, and duplicates
- Standardizing text fields
- Converting dates and numeric columns
- Translating product categories into English
- Creating delivery-performance metrics
- Validating table relationships
- Exporting analysis-ready CSV files

In [ ]:
## 1. Import Packages and Define File Paths

Pandas is used to load, clean, validate, and export the data.
Pathlib is used to identify the project folders.

In [2]:
# Import pandas for working with tables
import pandas as pd

# Import Path for working with project folders
from pathlib import Path


# Find the project root.
# Depending on VS Code, the notebook may start in either:
# - the main project folder, or
# - the python folder
current_location = Path.cwd()

if current_location.name == "python":
    project_root = current_location.parent
else:
    project_root = current_location


# Define the source and destination folders
raw_path = project_root / "data" / "raw"
cleaned_path = project_root / "data" / "cleaned"


# Confirm that the paths are correct
print("Current location:", current_location)
print("Project root:", project_root)
print("Raw data folder:", raw_path)
print("Raw folder exists:", raw_path.exists())

Current location: c:\Users\liset\Documents\GitHub\retail-sales-order-analytics\python
Project root: c:\Users\liset\Documents\GitHub\retail-sales-order-analytics
Raw data folder: c:\Users\liset\Documents\GitHub\retail-sales-order-analytics\data\raw
Raw folder exists: True


In [ ]:
## 2. Load the Raw Datasets

The project uses customer, order, item, payment, product, and
product-category data. The original CSV files remain unchanged
inside the `data/raw` folder.

In [4]:
# Load the customer table
customers = pd.read_csv(
    raw_path / "olist_customers_dataset.csv"
)

# Load the order table
orders = pd.read_csv(
    raw_path / "olist_orders_dataset.csv"
)

# Load the order-item table
order_items = pd.read_csv(
    raw_path / "olist_order_items_dataset.csv"
)

# Load the payment table
payments = pd.read_csv(
    raw_path / "olist_order_payments_dataset.csv"
)

# Load the product table
products = pd.read_csv(
    raw_path / "olist_products_dataset.csv"
)

# Load the Portuguese-to-English category translations
category_translation = pd.read_csv(
    raw_path / "product_category_name_translation.csv"
)


print("All six datasets loaded successfully.")

All six datasets loaded successfully.


In [ ]:
## 3. Inspect the Dataset Structure

The first profiling step confirms how many rows and columns are
contained in each table.

In [5]:
# Store all loaded tables in a dictionary.
# This allows us to inspect every table with one loop.
raw_tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "products": products,
    "category_translation": category_translation
}


# Create a list that will store the table-size results
table_sizes = []


# Count the rows and columns in each table
for table_name, dataframe in raw_tables.items():
    table_sizes.append({
        "table": table_name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1]
    })


# Convert the results into a DataFrame for a readable display
table_sizes = pd.DataFrame(table_sizes)

table_sizes

,table,rows,columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,payments,103886,5
4,products,32951,9
5,category_translation,71,2


In [ ]:
## 4. Assess Initial Data Quality

This section checks for exact duplicate rows and counts missing
values in each table. Missing values are evaluated before records
are changed or removed.

In [6]:
# Create a data-quality summary for every raw table
quality_profile = []


for table_name, dataframe in raw_tables.items():
    quality_profile.append({
        "table": table_name,

        # Count completely duplicated rows
        "duplicate_rows": dataframe.duplicated().sum(),

        # Count all missing cells in the table
        "total_missing_values": dataframe.isna().sum().sum(),

        # Count columns containing at least one missing value
        "columns_with_missing_values": (
            dataframe.isna().sum() > 0
        ).sum()
    })


quality_profile = pd.DataFrame(quality_profile)

quality_profile

,table,duplicate_rows,total_missing_values,columns_with_missing_values
0,customers,0,0,0
1,orders,0,4908,3
2,order_items,0,0,0
3,payments,0,0,0
4,products,0,2448,8
5,category_translation,0,0,0


In [7]:
# Display only the individual columns that contain missing values
missing_columns = []


for table_name, dataframe in raw_tables.items():
    for column in dataframe.columns:
        missing_count = dataframe[column].isna().sum()

        if missing_count > 0:
            missing_columns.append({
                "table": table_name,
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(dataframe) * 100,
                    2
                )
            })


missing_columns = pd.DataFrame(missing_columns)

missing_columns

,table,column,missing_count,missing_percent
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


In [ ]:
## 5. Create Working Copies

Cleaning is performed on copies of the original DataFrames. This
preserves the raw imported data for comparison and troubleshooting.

In [8]:
# Create separate working copies for cleaning
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
products_clean = products.copy()


print("Clean working copies created.")

Clean working copies created.


In [ ]:
## 6. Standardize Text Fields

Customer locations, order statuses, and payment methods are
standardized so identical values are not treated as separate
categories because of capitalization or extra spaces.

In [9]:
# ---------------------------------------------------------
# Customer locations
# ---------------------------------------------------------

# Remove extra spaces and capitalize city names
customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Remove extra spaces and capitalize state abbreviations
customers_clean["customer_state"] = (
    customers_clean["customer_state"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ---------------------------------------------------------
# Order statuses
# ---------------------------------------------------------

# Standardize order statuses as lowercase text
orders_clean["order_status"] = (
    orders_clean["order_status"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ---------------------------------------------------------
# Payment methods
# ---------------------------------------------------------

# Standardize payment methods as lowercase text
payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)


print("Text fields standardized.")

Text fields standardized.


In [10]:
# Preview the cleaned customer location fields
customers_clean[
    [
        "customer_city",
        "customer_state"
    ]
].head()

,customer_city,customer_state
0,Franca,SP
1,Sao Bernardo Do Campo,SP
2,Sao Paulo,SP
3,Mogi Das Cruzes,SP
4,Campinas,SP


In [ ]:
## 7. Convert Dates and Numeric Fields

Date columns must use a date-time data type before delivery
durations and monthly trends can be calculated. Monetary fields
are converted to numeric values for calculations.

In [11]:
# List the date columns contained in the orders table
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]


# Convert each order date from text to datetime
for column in order_date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )


# Convert the shipping deadline to datetime
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)


# Convert product price and freight cost to numeric values
order_items_clean["price"] = pd.to_numeric(
    order_items_clean["price"],
    errors="coerce"
)

order_items_clean["freight_value"] = pd.to_numeric(
    order_items_clean["freight_value"],
    errors="coerce"
)


# Convert payment information to numeric values
payments_clean["payment_value"] = pd.to_numeric(
    payments_clean["payment_value"],
    errors="coerce"
)

payments_clean["payment_installments"] = pd.to_numeric(
    payments_clean["payment_installments"],
    errors="coerce"
)


print("Date and numeric fields converted.")

Date and numeric fields converted.


In [12]:
# Verify that all order-date fields are datetime columns
orders_clean[order_date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [ ]:
## 8. Translate Product Categories

The original product categories are written in Portuguese. The
translation table is joined to the product table to provide
English category names for reports and dashboards.

In [13]:
# Join the translation table to the product table.
# "many_to_one" confirms that multiple products may share one
# category translation, but a category cannot have two translations.
products_clean = products_clean.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)


# Rename the translated category column
products_clean = products_clean.rename(
    columns={
        "product_category_name_english":
        "product_category_english"
    }
)


# Replace missing translations with "Unknown"
# and change underscores into spaces
products_clean["product_category_english"] = (
    products_clean["product_category_english"]
    .fillna("Unknown")
    .str.replace("_", " ", regex=False)
    .str.title()
)


# Preview the category translation
products_clean[
    [
        "product_id",
        "product_category_name",
        "product_category_english"
    ]
].head(10)

,product_id,product_category_name,product_category_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,Perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,Art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,Sports Leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,Baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,Housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,Musical Instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,Cool Stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,Furniture Decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,Home Appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,Toys


In [ ]:
## 9. Create Order-Delivery Metrics

Three business fields are created:

- Purchase date for time-based analysis
- Delivery duration in days
- Late-delivery indicator

Orders without an actual delivery date remain unclassified rather
than being incorrectly labeled as on time.

In [14]:
# Create a date-only purchase field for Power BI and SQL
orders_clean["order_purchase_date"] = (
    orders_clean["order_purchase_timestamp"].dt.date
)


# Calculate the number of days between purchase and delivery
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.days


# Start the late-delivery column with missing values
orders_clean["is_late"] = pd.NA


# Identify orders that have both actual and estimated delivery dates
valid_delivery_dates = (
    orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_estimated_delivery_date"].notna()
)


# Assign 1 to late orders and 0 to on-time or early orders
orders_clean.loc[
    valid_delivery_dates,
    "is_late"
] = (
    orders_clean.loc[
        valid_delivery_dates,
        "order_delivered_customer_date"
    ]
    >
    orders_clean.loc[
        valid_delivery_dates,
        "order_estimated_delivery_date"
    ]
).astype(int)


# Convert to a nullable integer:
# 1 = late, 0 = on time, missing = not delivered/evaluable
orders_clean["is_late"] = (
    orders_clean["is_late"].astype("Int64")
)


print("Delivery metrics created.")

Delivery metrics created.


In [15]:
# Preview the new operational fields
orders_clean[
    [
        "order_id",
        "order_status",
        "order_purchase_date",
        "delivery_days",
        "is_late"
    ]
].head(10)

,order_id,order_status,order_purchase_date,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02,8.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24,13.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08,9.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18,13.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13,2.0,0
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07-09,16.0,0
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2017-04-11,NaN,<NA>
7,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05-16,9.0,0
8,76c6e866289321a7c93b82b54852dc33,delivered,2017-01-23,9.0,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-07-29,18.0,0


In [16]:
# Count late, on-time, and unclassified orders
orders_clean["is_late"].value_counts(
    dropna=False
)

is_late
0       88649
1        7827
<NA>     2965
Name: count, dtype: Int64

In [ ]:
## 10. Validate the Cleaned Data

The final validation checks important identifiers, numeric values,
and table relationships before the cleaned files are exported.

In [17]:
# Create a focused list of data-quality checks
validation_results = pd.DataFrame({
    "quality_check": [
        "Duplicate customer IDs",
        "Duplicate order IDs",
        "Duplicate product IDs",
        "Negative product prices",
        "Negative freight values",
        "Negative payment values",
        "Negative delivery durations",
        "Orders without matching customers",
        "Order items without matching orders",
        "Payments without matching orders",
        "Order items without matching products"
    ],

    "problem_rows": [
        # Primary-key checks
        customers_clean["customer_id"].duplicated().sum(),
        orders_clean["order_id"].duplicated().sum(),
        products_clean["product_id"].duplicated().sum(),

        # Invalid numeric-value checks
        (order_items_clean["price"] < 0).sum(),
        (order_items_clean["freight_value"] < 0).sum(),
        (payments_clean["payment_value"] < 0).sum(),
        (orders_clean["delivery_days"] < 0).sum(),

        # Relationship checks
        (
            ~orders_clean["customer_id"]
            .isin(customers_clean["customer_id"])
        ).sum(),

        (
            ~order_items_clean["order_id"]
            .isin(orders_clean["order_id"])
        ).sum(),

        (
            ~payments_clean["order_id"]
            .isin(orders_clean["order_id"])
        ).sum(),

        (
            ~order_items_clean["product_id"]
            .isin(products_clean["product_id"])
        ).sum()
    ]
})


validation_results

,quality_check,problem_rows
0,Duplicate customer IDs,0
1,Duplicate order IDs,0
2,Duplicate product IDs,0
3,Negative product prices,0
4,Negative freight values,0
5,Negative payment values,0
6,Negative delivery durations,0
7,Orders without matching customers,0
8,Order items without matching orders,0
9,Payments without matching orders,0


In [ ]:
## 11. Export the Cleaned Data

The cleaned customer, order, order-item, payment, and product
tables are exported to `data/cleaned` for use in MySQL and Power BI.

In [18]:
# Create the destination folder if it does not already exist
cleaned_path.mkdir(
    parents=True,
    exist_ok=True
)


# Export each cleaned DataFrame without the pandas row index
customers_clean.to_csv(
    cleaned_path / "customers_clean.csv",
    index=False
)

orders_clean.to_csv(
    cleaned_path / "orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    cleaned_path / "order_items_clean.csv",
    index=False
)

payments_clean.to_csv(
    cleaned_path / "payments_clean.csv",
    index=False
)

products_clean.to_csv(
    cleaned_path / "products_clean.csv",
    index=False
)


print("Five cleaned datasets exported successfully.")

Five cleaned datasets exported successfully.
